# Xenium Prime 5K Breast — CCC_Phe CCC niches

The trained AE, P32/Z32 and H32 are reused. Final K is selected from cell-level program usage without retraining.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/phenoniche/pipeline.py').is_file())
SOURCE = ROOT / 'src'
if str(SOURCE) not in sys.path: sys.path.insert(0, str(SOURCE))
from phenoniche.pipeline import load_artifacts
from phenoniche.data import load_prepared_input
from phenoniche.reporting import (training_curves, model_selection_plot, spatial_overview,
    niche_celltype_enrichment, spatial_each_niche, niche_size_plot, representative_h_heatmap)
artifacts = load_artifacts()
summary, output, selected_K = artifacts['summary'], artifacts['output'], artifacts['selected_K']


## 1. Data and trained model

In [ ]:
pd.Series({'cells': summary['n_cells'], 'original CCC features': summary['n_input_features'],
           'retained CCC features': summary['n_retained_features'], 'latent programs': 32,
           'selected final K': selected_K, 'Architecture K': 8}, name='CCC_Phe')

In [ ]:
training_curves(output); plt.show()
pd.Series({'Stage 1 validation reconstruction': summary['stage1_validation_reconstruction'],
           'Stage 2 validation nonlinear AE': summary['stage2_validation_ae'],
           'Stage 2 validation Z32 H32': summary['stage2_validation_linear']})

## 2. Automatic K selection (4–15)

Clustering uses cell-level sqrt(P32) plus log CCC activity. Original Q8_direct is retained as the K=8 comparator.

In [ ]:
model_selection_plot(artifacts); plt.show()
columns = ['K','silhouette','adjacent_K_stability','max_niche_cell_fraction',
           'tiny_niche_fraction','mean_assignment_confidence','mean_H_program_separation',
           'selection_score','eligible','selected']
display(artifacts['model_selection'][columns].round(4))
print(f"Selected K={selected_K}: {summary['selection_reason']}")

## 3. Selected-K spatial map

In [ ]:
spatial_overview(artifacts); plt.show()

## 4. Niche size and assignment confidence

In [ ]:
niche_size_plot(artifacts); plt.show()
display(artifacts['counts'])
confidence = artifacts['assignments']['assignment_confidence_relative']
fig, ax = plt.subplots(figsize=(6.5, 3.2)); ax.hist(confidence, bins=40, color='#4c78a8')
ax.set(xlabel='Assignment confidence', ylabel='Cells', title=f'Selected K={selected_K} confidence')
figure_dir = output / 'figures'; figure_dir.mkdir(parents=True, exist_ok=True)
plt.tight_layout(); fig.savefig(figure_dir / 'assignment_confidence.png', dpi=300, bbox_inches='tight'); plt.show()
pd.Series(confidence).describe(percentiles=[.1,.25,.5,.75,.9]).to_frame('confidence')

## 5. Cell types and individual niche maps

Cell type is used only for post-training interpretation.

In [ ]:
_, enrichment_table = niche_celltype_enrichment(artifacts); plt.show()
print('Stars: one-sided Mann–Whitney U; dot size: across-niche entropy.')

In [ ]:
spatial_each_niche(artifacts); plt.show()

## 6. Representative CCC programs

In [ ]:
for niche in range(1, selected_K + 1):
    print(f'\nNiche {niche}')
    display(artifacts['top_ccc'].query('niche == @niche')[['rank','ccc','weight']].head(15).reset_index(drop=True))
selected_dir = output / 'selected_model'
print('Top sender → receiver')
display(pd.read_csv(selected_dir/'top_sender_receiver.csv').groupby('niche', group_keys=False).head(5))
print('Top ligand–receptor')
display(pd.read_csv(selected_dir/'top_lr.csv').groupby('niche', group_keys=False).head(5))

## 7. Selected niche × representative CCC heatmap

In [ ]:
prepared = load_prepared_input(artifacts['config'])
representative_h_heatmap(artifacts['H'], prepared.features); plt.show()

## 8. Latent-program contribution to selected niches

In [ ]:
compact = (artifacts['mapping'].groupby('final_niche_id')
           .agg(latent_programs=('program_id', lambda x: ', '.join(map(str, x))),
                total_exposure=('mean_exposure','sum')).reset_index())
display(compact)

## 9. Selected K versus original Q8_direct comparator

In [ ]:
display(pd.DataFrame({
    'niches': [8, selected_K],
    'largest_niche_fraction': [summary['K8_max_niche_cell_fraction'], summary['selected_max_niche_cell_fraction']],
    'tiny_niche_fraction': [summary['K8_tiny_niche_fraction'], summary['selected_tiny_niche_fraction']],
    'mean_assignment_confidence': [summary['K8_mean_assignment_confidence'], summary['selected_mean_assignment_confidence']]
}, index=['Original Q8_direct', f'Cell-latent selected K={selected_K}']).round(4))

## 10. Summary

In [ ]:
top = artifacts['top_ccc'].query('rank == 1').set_index('niche')['ccc'].to_dict()
print(f'Selected K: {selected_K}; all selected niches used: {len(artifacts["counts"]) == selected_K and (artifacts["counts"].n_cells > 0).all()}')
print(f'Largest selected niche: {summary["selected_max_niche_cell_fraction"]:.2%}')
print(f'Tiny-niche cell fraction: {summary["selected_tiny_niche_fraction"]:.2%}')
print(f'Mean assignment confidence: {summary["selected_mean_assignment_confidence"]:.3f}')
display(pd.Series(top, name='Top representative CCC').rename_axis('Niche').to_frame())